In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import f1_score
from scipy.stats import mode

In [2]:
base_meta_path = '/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/metadata'
pre2020_pred_path = '/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/Github/IAV_Evolution/results/pre2020_calibration'
post2020_pred_path = '/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/Github/IAV_Evolution/results/post2020_calibration'

ground_truth_col = 'Host_ID'  # 0: Avian, 1: Human
num_folds = 10 

# Target Subtypes for Reporting
target_subtypes = ['H1N1', 'H3N2', 'H1N2', 'H2N2', 'H7N7', 'H3N8', 'H9N2', 'H7N9', 'H5N1', 'H5N6']

segments = [
    ("01", "PB2"), ("02", "PB1"), ("03", "PA"), ("04", "HA"),
    ("05", "NP"), ("06", "NA"), ("07", "MP"), ("08", "NS")
]

# Helper Function for Reporting
def generate_failure_report(df, dataset_name):
    """Generates and prints failure statistics for a given dataframe."""
    failed_df = df[df['is_failed'] == True].copy()
    
    if failed_df.empty:
        print(f"  [{dataset_name}] Status: PERFECT (100% Accuracy)")
        return
    
    print(f"  [{dataset_name}] Status: {len(failed_df)} Failures")
    
    # Group failures
    failure_summary = failed_df.groupby(
        ['Subtype', ground_truth_col, 'pred_label']
    ).size().reset_index(name='Fail_Count')

    # Group totals
    total_counts = df.groupby(['Subtype', ground_truth_col]).size().reset_index(name='Total_Count')
    
    # Merge
    report = pd.merge(failure_summary, total_counts, on=['Subtype', ground_truth_col], how='left')
    report['Pass_Count'] = report['Total_Count'] - report['Fail_Count']
    report['Fail_Rate (%)'] = (report['Fail_Count'] / report['Total_Count'] * 100).round(2)
    
    # Filter Subtypes
    report = report[report['Subtype'].isin(target_subtypes)]
    #print (report[report['Subtype'].isin(['H2N2'])])
    
    if not report.empty:
        # Reorder for clarity
        cols = ['Subtype', ground_truth_col, 'pred_label', 'Pass_Count', 'Fail_Count', 'Total_Count', 'Fail_Rate (%)']
        print(f"\n  [{dataset_name} Failure Report]")
        print(report[cols].sort_values('Subtype').to_string(index=False))
    else:
        print(f"  [{dataset_name}] No failures in target subtypes.")

# ==========================================
# 2. MAIN LOOP
# ==========================================
for num, seg in segments:
    print(f"\n{'='*60}")
    print(f"Processing Segment: {seg}")
    print(f"{'='*60}")

    # ---------------------------------------------------------
    # PART A: PRE-2020 (Single Model Training Set)
    # ---------------------------------------------------------
    csv_pre = os.path.join(base_meta_path, f"{num}_{seg}", f"{num}_{seg}_Train_Filtered.csv")
    npy_pre = os.path.join(pre2020_pred_path, f"{num}_{seg}_preds_cal_fold_0.npy")
    
    if os.path.exists(csv_pre) and os.path.exists(npy_pre):
        df_pre = pd.read_csv(csv_pre)
        preds_pre = np.load(npy_pre)
        
        if preds_pre.ndim > 1: preds_pre = preds_pre.ravel()
        
        if len(df_pre) == len(preds_pre):
            df_pre['pred_label'] = preds_pre.astype(int)
            
            # Metrics
            f1 = f1_score(df_pre[ground_truth_col], df_pre['pred_label'], average='macro')
            print(f"  [Pre-2020] F1-Score (Macro): {f1:.4f}")
            
            # Failures
            df_pre['is_failed'] = df_pre['pred_label'] != df_pre[ground_truth_col].astype(int)
            generate_failure_report(df_pre, "Pre-2020")
        else:
            print(f"  [Pre-2020] Error: Length Mismatch ({len(df_pre)} vs {len(preds_pre)})")
    else:
        print(f"  [Pre-2020] Skip (Missing files)")

    print("-" * 40)

    # ---------------------------------------------------------
    # PART B: POST-2020 (10-Fold Ensemble Test Set)
    # ---------------------------------------------------------
    csv_post = os.path.join(base_meta_path, f"{num}_{seg}", f"Part1_2_{num}_{seg}_Test_Filtered.csv")
    
    if os.path.exists(csv_post):
        df_post = pd.read_csv(csv_post)
        
        # Load 10 Folds
        fold_preds = []
        for i in range(num_folds):
            npy_post = os.path.join(post2020_pred_path, f"{num}_{seg}_preds_cal_fold_{i}.npy")
            if os.path.exists(npy_post):
                arr = np.load(npy_post)
                if arr.ndim > 1: arr = arr.ravel()
                if len(arr) == len(df_post):
                    fold_preds.append(arr.astype(int))
        
        if fold_preds:
            # Majority Vote Ensemble
            stack = np.vstack(fold_preds)
            majority_vote, _ = mode(stack, axis=0, keepdims=False)
            df_post['pred_label'] = majority_vote
            
            # Metrics
            f1_post = f1_score(df_post[ground_truth_col], df_post['pred_label'], average='macro')
            print(f"  [Post-2020] Ensemble F1-Score (Macro): {f1_post:.4f} (from {len(fold_preds)} folds)")
            
            # Failures
            df_post['is_failed'] = df_post['pred_label'] != df_post[ground_truth_col].astype(int)
            generate_failure_report(df_post, "Post-2020")
        else:
            print("  [Post-2020] Skip (No valid prediction files found)")
    else:
        print(f"  [Post-2020] Skip (Missing Metadata CSV)")

print(f"\n{'='*60}\nDone.")



Processing Segment: PB2
  [Pre-2020] F1-Score (Macro): 0.9502
  [Pre-2020] Status: 2774 Failures

  [Pre-2020 Failure Report]
Subtype  Host_ID  pred_label  Pass_Count  Fail_Count  Total_Count  Fail_Rate (%)
   H1N1        0           1       20203           8        20211           0.04
   H1N1        0           2       20014         197        20211           0.97
   H1N1        1           0         542          15          557           2.69
   H1N1        1           2         553           4          557           0.72
   H1N1        2           0        2812         533         3345          15.93
   H1N1        2           1        3313          32         3345           0.96
   H1N2        2           1        2368          18         2386           0.75
   H1N2        2           0        2248         138         2386           5.78
   H1N2        1           2          56           1           57           1.75
   H1N2        0           2          35          21           